In [ ]:
# In[1]:

import pandas as pd
import pytz

# Reuse timezone variable per rule 10 (UTC+8), though timestamps remain integers here
tz = pytz.timezone('Asia/Shanghai')

# Filenames (files are in current working directory)
f_metric = "metric.csv"
f_trace = "trace.csv"
f_log = "log.csv"
f_err = "error_logs.csv"

# Load CSVs
df_metric = pd.read_csv(f_metric)
df_trace = pd.read_csv(f_trace)
df_log = pd.read_csv(f_log)
df_err = pd.read_csv(f_err)

# Ensure timestamp ints and numeric values where applicable
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_log['timestamp'] = df_log['timestamp'].astype(int)
df_err['timestamp'] = df_err['timestamp'].astype(int)

df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# 1) metric.csv: group by cmdb_id and kpi_name
metric_summary = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count_points=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95_value=('value', lambda s: s.quantile(0.95)),
        median_value=('value', 'median')
    )
    .sort_values('count_points', ascending=False)
    .head(20)
)

# 2) trace.csv: group by cmdb_id and trace_name
trace_summary = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count_points=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95_value=('value', lambda s: s.quantile(0.95)),
        median_value=('value', 'median')
    )
    .sort_values('count_points', ascending=False)
    .head(20)
)

# 3) log.csv: group by cmdb_id and log_name
log_summary = (
    df_log
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        count_points=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95_value=('value', lambda s: s.quantile(0.95)),
        median_value=('value', 'median')
    )
    .sort_values('count_points', ascending=False)
    .head(20)
)

# 4) error_logs.csv: group by cmdb_id (count, min, max)
error_summary = (
    df_err
    .groupby('cmdb_id', as_index=False)
    .agg(
        count_points=('timestamp', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max')
    )
    .sort_values('count_points', ascending=False)
    .head(20)
)

# Round numeric thresholds for compactness
for df in (metric_summary, trace_summary, log_summary):
    if 'p95_value' in df.columns:
        df['p95_value'] = df['p95_value'].round(6)
    if 'median_value' in df.columns:
        df['median_value'] = df['median_value'].round(6)

# Return the four summaries (compact, top 20 each)
metric_summary, trace_summary, log_summary, error_summary

```
Out[1]:
```
```python
# Build a plain-English summary string using previous aggregation results
# Reuse metric_summary, trace_summary, log_summary, error_summary from the kernel state.

def fmt_num(x):
    if pd.isna(x):
        return "NA"
    if isinstance(x, (int,)):
        return f"{x:,}"
    try:
        # decide decimals
        if abs(x) >= 1e6 or (isinstance(x, float) and x.is_integer() and abs(x) >= 1000):
            return f"{int(round(x)):,}"
        if abs(x) >= 1:
            return f"{x:,.3f}".rstrip('0').rstrip('.')
        return f"{x:.6f}".rstrip('0').rstrip('.')
    except Exception:
        return str(x)

lines = []

# Metric summary top rows (up to 3)
lines.append("Metrics (top groups by point count):")
if not metric_summary.empty:
    for i, row in metric_summary.head(3).iterrows():
        lines.append(
            f"- {row['cmdb_id']} / {row['kpi_name']}: {int(row['count_points'])} points, "
            f"time range [{int(row['min_timestamp'])} .. {int(row['max_timestamp'])}], "
            f"P95={fmt_num(row['p95_value'])}, median={fmt_num(row['median_value'])}"
        )
else:
    lines.append("- No metric groups found.")

# Trace summary top rows (up to 3)
lines.append("\nTraces (top groups by point count):")
if not trace_summary.empty:
    for i, row in trace_summary.head(3).iterrows():
        lines.append(
            f"- {row['cmdb_id']} / {row['trace_name']}: {int(row['count_points'])} points, "
            f"time range [{int(row['min_timestamp'])} .. {int(row['max_timestamp'])}], "
            f"P95={fmt_num(row['p95_value'])}, median={fmt_num(row['median_value'])}"
        )
else:
    lines.append("- No trace groups found.")

# Log summary top rows (up to 4)
lines.append("\nLogs (top groups by point count):")
if not log_summary.empty:
    for i, row in log_summary.head(4).iterrows():
        lines.append(
            f"- {row['cmdb_id']} / {row['log_name']}: {int(row['count_points'])} points, "
            f"time range [{int(row['min_timestamp'])} .. {int(row['max_timestamp'])}], "
            f"P95={fmt_num(row['p95_value'])}, median={fmt_num(row['median_value'])}"
        )
else:
    lines.append("- No log groups found.")

# Error logs summary
lines.append("\nError logs (per cmdb_id):")
if not error_summary.empty:
    for i, row in error_summary.iterrows():
        lines.append(
            f"- {row['cmdb_id']}: {int(row['count_points'])} entries, "
            f"time range [{int(row['min_timestamp'])} .. {int(row['max_timestamp'])}]"
        )
else:
    lines.append("- No error_log entries found (empty).")

summary = "\n".join(lines)

# Display the concise plain-English summary
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count_points  min_timestamp  max_timestamp     p95_value  median_value
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads            31     1615266000     1615267800  7.200000e+01  6.900000e+01
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed            31     1615266000     1615267800  5.790867e+07  5.790867e+07
1657  apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            31     1615266000     1615267800  0.000000e+00  0.000000e+00
1658  apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            31     1615266000     1615267800  0.000000e+00  0.000000e+00
2         IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed            31     1615266000     1615267800  8.484594e+08  5.159736e+08
1660  apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies            31     1615266000     1615267800  0.000000e+00  0.000000e+00
720    Mysql02                         OSLinux-CPU_CPU_CPUSysTime            31     1615266000     1615267800  1.617250e+00  1.334276e+00
721    Mysql02                        OSLinux-CPU_CPU_CPUUserTime            31     1615266000     1615267800  2.494060e+00  1.935400e+00
722    Mysql02                             OSLinux-CPU_CPU_CPUWio            31     1615266000     1615267800  1.686900e+01  1.087590e+01
723    Mysql02                        OSLinux-CPU_CPU_CPUidleutil            31     1615266000     1615267800  8.758260e+01  8.559643e+01
719    Mysql02                            OSLinux-CPU_CPU_CPULoad            31     1615266000     1615267800  1.340000e+00  8.000000e-01
802    Mysql02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...            31     1615266000     1615267800  6.000000e-01  6.000000e-01
803    Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31     1615266000     1615267800  2.400000e+01  2.400000e+01
796    Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            31     1615266000     1615267800  0.000000e+00  0.000000e+00
797    Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            31     1615266000     1615267800  0.000000e+00  0.000000e+00
799    Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies            31     1615266000     1615267800  0.000000e+00  0.000000e+00
816    Redis01                         OSLinux-CPU_CPU_CPUSysTime            31     1615266000     1615267800  1.872950e+00  1.627900e+00
817    Redis01                        OSLinux-CPU_CPU_CPUUserTime            31     1615266000     1615267800  2.524102e+01  2.516440e+01
818    Redis01                             OSLinux-CPU_CPU_CPUWio            31     1615266000     1615267800  1.916000e-01  1.249500e-02
819    Redis01                        OSLinux-CPU_CPU_CPUidleutil            31     1615266000     1615267800  7.257814e+01  7.204910e+01,    cmdb_id                       trace_name  count_points  min_timestamp  max_timestamp    p95_value  median_value
0     IG01         trace.self.duration_mean            30     1615266000     1615267740     0.583523      0.441486
1     IG01          trace.self.duration_p95            30     1615266000     1615267740     2.887780      2.088075
2     IG01             trace.self.row_count            30     1615266000     1615267740  1533.100000   1382.000000
3     IG01  trace.to_Tomcat01.duration_mean            30     1615266000     1615267740     0.597473      0.439858
4     IG01   trace.to_Tomcat01.duration_p95            30     1615266000     1615267740     2.854822      2.076875
5     IG01      trace.to_Tomcat01.row_count            30     1615266000     1615267740   205.550000    174.500000
6     IG01  trace.to_Tomcat02.duration_mean            30     1615266000     1615267740     0.582446      0.438826
7     IG01   trace.to_Tomcat02.duration_p95            30     1615266000     1615267740     3.117085      2.078375
8     IG01      trace.to_Tomcat02.row_count            30     1615266000     1615267740   204.600000    174.000000
9     IG01  trace.to_Tomcat03.duration_mean            30     1615266000     1615267740     0.655579      0.434301
10    IG01   trace.to_Tomcat03.duration_p95            30     1615266000     1615267740     3.028707      2.089650
11    IG01      trace.to_Tomcat03.row_count            30     1615266000     1615267740   205.550000    172.500000
12    IG01  trace.to_Tomcat04.duration_mean            30     1615266000     1615267740     0.618188      0.462432
13    IG01   trace.to_Tomcat04.duration_p95            30     1615266000     1615267740     3.125360      2.090700
14    IG01      trace.to_Tomcat04.row_count            30     1615266000     1615267740   189.750000    170.500000
15    IG02         trace.self.duration_mean            30     1615266000     1615267740     0.676502      0.438568
16    IG02          trace.self.duration_p95            30     1615266000     1615267740     2.847722      2.092500
17    IG02             trace.self.row_count            30     1615266000     1615267740  1504.100000   1344.000000
18    IG02  trace.to_Tomcat01.duration_mean            30     1615266000     1615267740     0.664577      0.431102
19    IG02   trace.to_Tomcat01.duration_p95            30     1615266000     1615267740     2.618905      2.074000,      cmdb_id          log_name  count_points  min_timestamp  max_timestamp  p95_value  median_value
4   Tomcat01     log.row_count            31     1615266000     1615267800    392.000        339.00
2   Tomcat01   log.error_count            31     1615266000     1615267800      0.000          0.00
17  apache02     log.row_count            31     1615266000     1615267800    768.000        676.00
15  apache01     log.row_count            31     1615266000     1615267800    768.000        686.00
8   Tomcat03   log.error_count            31     1615266000     1615267800      0.000          0.00
16  apache02   log.error_count            31     1615266000     1615267800      0.000          0.00
14  apache01   log.error_count            31     1615266000     1615267800      0.000          0.00
13  Tomcat04     log.row_count            31     1615266000     1615267800    390.000        341.00
11  Tomcat04   log.error_count            31     1615266000     1615267800      0.000          0.00
10  Tomcat03     log.row_count            31     1615266000     1615267800    396.500        337.00
1    Mysql01     log.row_count            20     1615266000     1615267680      7.450          3.00
0    Mysql01   log.error_count            20     1615266000     1615267680      0.000          0.00
3   Tomcat01  log.gc_pause_max            18     1615266000     1615267680      0.106          0.05
9   Tomcat03  log.gc_pause_max            17     1615266060     1615267680      0.092          0.05
12  Tomcat04  log.gc_pause_max            17     1615266000     1615267680      0.090          0.08
6   Tomcat02  log.gc_pause_max            15     1615266000     1615267680      0.093          0.08
5   Tomcat02   log.error_count            15     1615266000     1615267680      0.000          0.00
7   Tomcat02     log.row_count            15     1615266000     1615267680      1.000          1.00, Empty DataFrame
Columns: [cmdb_id, count_points, min_timestamp, max_timestamp]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from datetime import datetime, timedelta

# Reuse existing dataframes if present in kernel; otherwise load from CSV
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = df_metric['timestamp'].astype(int)
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Ensure timestamp and value types
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Define incident window in UTC+8: Mar 9, 2021 13:00 to 13:30 (inclusive)
start_local = tz.localize(datetime(2021, 3, 9, 13, 0, 0))
end_local   = tz.localize(datetime(2021, 3, 9, 13, 30, 0))

start_epoch = int(start_local.timestamp())
end_epoch   = int(end_local.timestamp())

# Rule 9: compute global thresholds (P95/median) from the full metric series BEFORE filtering
metric_global = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        global_count=('value', 'count'),
        p95_value=('value', lambda s: s.quantile(0.95)),
        median_value=('value', 'median')
    )
)

# Filter full metric series to the incident time window and store in kernel variable metric_window
metric_window = df_metric[(df_metric['timestamp'] >= start_epoch) & (df_metric['timestamp'] <= end_epoch)].copy()

# Helper to compute anomaly summary given a threshold column value (use p95_value first, fallback to p90)
def compute_anomalies(threshold_quantile=0.95):
    # If threshold_quantile != 0.95, compute appropriate global threshold per group
    if threshold_quantile == 0.95:
        thresholds = metric_global[['cmdb_id','kpi_name','p95_value','median_value']].copy()
    else:
        # compute p90 from full series
        t = df_metric.groupby(['cmdb_id','kpi_name'])['value'].quantile(threshold_quantile).reset_index(name='pXX_value')
        med = df_metric.groupby(['cmdb_id','kpi_name'])['value'].median().reset_index(name='median_value')
        thresholds = pd.merge(t, med, on=['cmdb_id','kpi_name'], how='left')
        thresholds = thresholds.rename(columns={'pXX_value':'p95_value'})  # use column name p95_value generically
    
    # Join thresholds to windowed points
    mw = metric_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left', suffixes=('','_th'))
    # mark anomalies where value >= threshold (larger is worse)
    mw['is_anom'] = mw['value'] >= mw['p95_value']
    
    if mw.shape[0] == 0:
        # no data in window at all
        return pd.DataFrame(columns=[
            'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp',
            'max_value_in_window','count_points_in_window','p95_value','median_value'
        ])
    
    # aggregate per group
    agg = mw.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        anomaly_count=('is_anom','sum'),
        earliest_anomaly_timestamp=('timestamp', lambda s: int(mw.loc[(mw['is_anom']) & (mw.index.isin(s.index)), 'timestamp'].min()) if any(mw.loc[s.index,'is_anom']) else pd.NA),
        max_value_in_window=('value','max'),
        count_points_in_window=('value','count'),
        p95_value=('p95_value','first'),
        median_value=('median_value','first')
    )
    # The lambda for earliest_anomaly_timestamp is a bit complex; recompute more directly:
    # recompute earliest_anomaly_timestamp correctly per group
    def get_earliest(row):
        grp = mw[(mw['cmdb_id']==row['cmdb_id']) & (mw['kpi_name']==row['kpi_name']) & (mw['is_anom'])]
        if grp.empty:
            return pd.NA
        return int(grp['timestamp'].min())
    if not agg.empty:
        agg['earliest_anomaly_timestamp'] = agg.apply(get_earliest, axis=1)
    
    # Only keep groups with anomaly_count >=1
    agg = agg[agg['anomaly_count'] > 0].copy()
    
    # Round p95/median for compactness
    agg['p95_value'] = agg['p95_value'].round(6)
    agg['median_value'] = agg['median_value'].round(6)
    
    # Order by anomaly_count desc, limit 20
    agg = agg.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
    return agg

# First attempt using P95
anomaly_summary = compute_anomalies(0.95)

# If no anomalies found, retry with P90
if anomaly_summary.shape[0] == 0:
    anomaly_summary = compute_anomalies(0.90)

# Final compact display: selected columns and top 20 already ensured
anomaly_summary

```
Out[2]:
```
```python
# Build a plain-English summary from the existing anomaly_summary variable in the kernel.
# Reuse anomaly_summary and metric_window computed earlier.

import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

# Defensive: ensure anomaly_summary exists
try:
    anomaly_summary
except NameError:
    summary = "No anomaly_summary found in kernel."
else:
    n_groups = len(anomaly_summary)
    top_n = min(20, n_groups)
    
    def fmt_num(x):
        if pd.isna(x):
            return "NA"
        try:
            if float(x).is_integer():
                return f"{int(x):,}"
        except Exception:
            pass
        x = float(x)
        if abs(x) >= 1000:
            return f"{x:,.0f}"
        if abs(x) >= 1:
            return f"{x:,.3f}".rstrip('0').rstrip('.')
        return f"{x:.6f}".rstrip('0').rstrip('.')
    
    def to_local_ts(epoch):
        if pd.isna(epoch):
            return "NA"
        dt = datetime.fromtimestamp(int(epoch), tz)
        return dt.strftime("%Y-03-09 %H:%M:%S %Z") if dt.year==2021 and dt.month==3 and dt.day==9 else dt.strftime("%Y-%m-%d %H:%M:%S %Z")
    
    lines = []
    lines.append(f"Detected {n_groups} metric group(s) with anomalies in the window Mar 9, 2021 13:00–13:30 (UTC+8). Showing top {top_n} groups ordered by anomaly count:")
    for _, row in anomaly_summary.head(top_n).iterrows():
        cmdb = row['cmdb_id']
        kpi = row['kpi_name']
        ac = int(row['anomaly_count'])
        earliest_epoch = row['earliest_anomaly_timestamp']
        earliest_hr = to_local_ts(earliest_epoch)
        max_val = fmt_num(row['max_value_in_window'])
        points = int(row['count_points_in_window'])
        p95 = fmt_num(row['p95_value'])
        med = fmt_num(row['median_value'])
        lines.append(f"- {cmdb} / {kpi}: anomalies={ac}, earliest={earliest_epoch} ({earliest_hr}), max_in_window={max_val}, points_in_window={points}, global_P95={p95}, median={med}")
    
    summary = "\n".join(lines)

# Display the concise plain-English summary
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_timestamp  max_value_in_window  count_points_in_window     p95_value  median_value
0        Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31                  1615266000         2.400000e+01                      31  2.400000e+01  2.400000e+01
1       apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31                  1615266000         4.000000e-01                      31  4.000000e-01  4.000000e-01
2       apache02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31                  1615266000         7.000000e+00                      31  7.000000e+00  7.000000e+00
3           IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31                  1615266000         2.040136e+09                      31  2.040136e+09  2.040136e+09
4        Redis01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31                  1615266000         1.000000e+00                      31  1.000000e+00  1.000000e+00
5        Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31                  1615266000         0.000000e+00                      31  0.000000e+00  0.000000e+00
6        Redis01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31                  1615266000         4.000000e+00                      31  4.000000e+00  4.000000e+00
7       Tomcat02  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31                  1615266000         0.000000e+00                      31  0.000000e+00  0.000000e+00
8       Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31                  1615266000         0.000000e+00                      31  0.000000e+00  0.000000e+00
9        Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31                  1615266000         0.000000e+00                      31  0.000000e+00  0.000000e+00
10          IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31                  1615266000         0.000000e+00                      31  0.000000e+00  0.000000e+00
11  ServiceTest7                                             app.sr             31                  1615266000         1.000000e+02                      31  1.000000e+02  1.000000e+02
12       Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31                  1615266000         0.000000e+00                      31  0.000000e+00  0.000000e+00
13      Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31                  1615266000         1.000000e+02                      31  1.000000e+02  1.000000e+02
14      Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31                  1615266000         6.430000e+02                      31  6.430000e+02  6.430000e+02
15      Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31                  1615266000         4.053960e+05                      31  4.053960e+05  4.053960e+05
16      Tomcat01    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31                  1615266000         0.000000e+00                      31  0.000000e+00  0.000000e+00
17  ServiceTest8                                             app.sr             31                  1615266000         1.000000e+02                      31  1.000000e+02  1.000000e+02
18  ServiceTest9                                             app.sr             31                  1615266000         1.000000e+02                      31  1.000000e+02  1.000000e+02
19  ServiceTest6                                             app.sr             31                  1615266000         1.000000e+02                      31  1.000000e+02  1.000000e+02```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing df_trace and trace_summary if present; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['timestamp'] = df_trace['timestamp'].astype(int)
    df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Ensure types
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# UTC+8 timezone
tz = pytz.timezone('Asia/Shanghai')

# Incident window (UTC+8): Mar 9, 2021 13:00 to 13:30
start_local = tz.localize(datetime(2021, 3, 9, 13, 0, 0))
end_local = tz.localize(datetime(2021, 3, 9, 13, 30, 0))
start_epoch = int(start_local.timestamp())
end_epoch = int(end_local.timestamp())

# Rule 9: compute global thresholds (P95 and median) from full trace series BEFORE filtering
trace_global = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        p95_value=('value', lambda s: s.quantile(0.95)),
        median_value=('value', 'median')
    )
)

# Filter trace rows inside the time window and store in kernel variable trace_window
trace_window = df_trace[(df_trace['timestamp'] >= start_epoch) & (df_trace['timestamp'] <= end_epoch)].copy()

def compute_trace_anomalies(global_thresholds, quantile_label='p95_value'):
    # Merge windowed rows with global thresholds
    tw = trace_window.merge(global_thresholds, on=['cmdb_id', 'trace_name'], how='left')
    # Mark anomalies where value >= threshold (larger is worse)
    tw['is_anom'] = tw['value'] >= tw[quantile_label]
    
    if tw.empty:
        # No data in window
        return pd.DataFrame(columns=[
            'cmdb_id','trace_name','anomaly_count','earliest_anomaly_timestamp',
            'max_value_in_window','count_points_in_window','p95_value','median_value'
        ])
    
    # Per-group aggregates
    grp = tw.groupby(['cmdb_id','trace_name'], as_index=False)
    agg_counts = grp.agg(
        anomaly_count=('is_anom','sum'),
        max_value_in_window=('value','max'),
        count_points_in_window=('value','count')
    )
    # earliest anomaly timestamp per group (only from anomalous rows)
    anom_ts = tw[tw['is_anom']].groupby(['cmdb_id','trace_name'])['timestamp'].min().reset_index(name='earliest_anomaly_timestamp')
    # attach global thresholds (use columns from global_thresholds)
    thresholds = global_thresholds.rename(columns={quantile_label:'p95_value'})[['cmdb_id','trace_name','p95_value','median_value']]
    
    summary = agg_counts.merge(anom_ts, on=['cmdb_id','trace_name'], how='left')
    summary = summary.merge(thresholds, on=['cmdb_id','trace_name'], how='left')
    
    # Keep only groups with anomaly_count > 0
    summary = summary[summary['anomaly_count'] > 0].copy()
    if summary.empty:
        return summary
    
    # Round numeric thresholds for compactness
    summary['p95_value'] = summary['p95_value'].round(6)
    summary['median_value'] = summary['median_value'].round(6)
    
    # Order by anomaly_count desc and limit to top 20
    summary = summary.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
    return summary

# First attempt using global P95 from trace_global
anomaly_summary = compute_trace_anomalies(trace_global, quantile_label='p95_value')

# If no anomalies found, compute P90 thresholds and retry
if anomaly_summary.shape[0] == 0:
    trace_global_p90 = (
        df_trace
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(
            p90_value=('value', lambda s: s.quantile(0.90)),
            median_value=('value','median')
        )
    )
    # rename p90 to p95_value slot for function compatibility
    trace_global_p90 = trace_global_p90.rename(columns={'p90_value':'p95_value'})
    anomaly_summary = compute_trace_anomalies(trace_global_p90, quantile_label='p95_value')

# Return compact table (anomaly_summary). trace_window is stored for later use.
anomaly_summary

```
Out[3]:
```
```python
# Build a plain-English summary of the trace anomaly detection results (use existing anomaly_summary)
import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

# Defensive access to anomaly_summary
try:
    anomaly_summary
except NameError:
    summary = "No trace anomaly summary found in the kernel."
else:
    n_groups = len(anomaly_summary)
    top_n = min(20, n_groups)
    
    def fmt_num(x):
        if pd.isna(x):
            return "NA"
        try:
            xf = float(x)
        except Exception:
            return str(x)
        if abs(xf) >= 1000:
            return f"{xf:,.0f}"
        if abs(xf) >= 1:
            s = f"{xf:,.3f}".rstrip('0').rstrip('.')
            return s
        return f"{xf:.6f}".rstrip('0').rstrip('.')
    
    def to_local(epoch):
        if pd.isna(epoch):
            return "NA"
        dt = datetime.fromtimestamp(int(epoch), tz)
        return dt.strftime("%Y-%m-%d %H:%M:%S %Z")
    
    lines = []
    lines.append(f"Trace anomaly detection (window Mar 9, 2021 13:00–13:30 UTC+8): {n_groups} trace group(s) had values >= global P95.")
    lines.append(f"Showing top {top_n} groups ordered by anomaly_count:")
    
    for _, row in anomaly_summary.head(top_n).iterrows():
        cmdb = row['cmdb_id']
        tname = row['trace_name']
        ac = int(row['anomaly_count'])
        earliest = to_local(row.get('earliest_anomaly_timestamp'))
        maxv = fmt_num(row.get('max_value_in_window'))
        pts = int(row.get('count_points_in_window', 0))
        p95 = fmt_num(row.get('p95_value'))
        med = fmt_num(row.get('median_value'))
        lines.append(f"- {cmdb} / {tname}: anomalies={ac}, earliest={earliest}, max_in_window={maxv}, points_in_window={pts}, global_P95={p95}, median={med}")
    
    summary = "\n".join(lines)

# Display the concise plain-English summary
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                        trace_name  anomaly_count  max_value_in_window  count_points_in_window  earliest_anomaly_timestamp    p95_value  median_value
0   Tomcat02           trace.self.duration_p95              4             1.005000                      30                  1615266900     1.005000      0.061150
1   dockerA2           trace.self.duration_p95              4             1.017000                      30                  1615266480     1.017000      1.015000
2   dockerA1           trace.self.duration_p95              3             0.013700                      30                  1615266480     0.010000      0.009000
3   Tomcat04           trace.self.duration_p95              3             1.005000                      30                  1615267080     1.005000      0.057450
4   dockerB2        trace.to_MG02.duration_p95              3             1.110900                      28                  1615266000     1.015000      0.319950
5       MG02  trace.from_dockerB2.duration_p95              3             1.110900                      28                  1615266000     1.015000      0.319950
6       IG01           trace.self.duration_p95              2             3.165000                      30                  1615267080     2.887780      2.088075
7       IG01   trace.to_Tomcat01.duration_mean              2             0.625989                      30                  1615266180     0.597473      0.439858
8       IG01              trace.self.row_count              2          1630.000000                      30                  1615266120  1533.100000   1382.000000
9       IG01   trace.to_Tomcat03.duration_mean              2             0.883940                      30                  1615267080     0.655579      0.434301
10      IG01    trace.to_Tomcat03.duration_p95              2             4.171600                      30                  1615266720     3.028707      2.089650
11      IG01       trace.to_Tomcat03.row_count              2           212.000000                      30                  1615266120   205.550000    172.500000
12      IG01   trace.to_Tomcat04.duration_mean              2             0.664446                      30                  1615267020     0.618188      0.462432
13      IG01    trace.to_Tomcat04.duration_p95              2             3.169000                      30                  1615267020     3.125360      2.090700
14      IG01       trace.to_Tomcat04.row_count              2           193.000000                      30                  1615267080   189.750000    170.500000
15      IG02          trace.self.duration_mean              2             0.745752                      30                  1615266600     0.676502      0.438568
16      IG02           trace.self.duration_p95              2             3.137000                      30                  1615267080     2.847722      2.092500
17      IG02              trace.self.row_count              2          1634.000000                      30                  1615266900  1504.100000   1344.000000
18      IG02   trace.to_Tomcat01.duration_mean              2             0.960076                      30                  1615266600     0.664577      0.431102
19      IG02    trace.to_Tomcat01.duration_p95              2             3.121350                      30                  1615267080     2.618905      2.074000```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse existing kernel variables
try:
    metric_window
    metric_summary
except NameError:
    # fallback load if not present (defensive)
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = df_metric['timestamp'].astype(int)
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    tz = pytz.timezone('Asia/Shanghai')
    start_local = tz.localize(__import__('datetime').datetime(2021,3,9,13,0,0))
    end_local = tz.localize(__import__('datetime').datetime(2021,3,9,13,30,0))
    start_epoch = int(start_local.timestamp())
    end_epoch = int(end_local.timestamp())
    metric_window = df_metric[(df_metric['timestamp'] >= start_epoch) & (df_metric['timestamp'] <= end_epoch)].copy()
    metric_summary = (
        df_metric
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(p95_value=('value', lambda s: s.quantile(0.95)), median_value=('value','median'))
    )

# Ensure types
metric_window['timestamp'] = metric_window['timestamp'].astype(int)
metric_window['value'] = pd.to_numeric(metric_window['value'], errors='coerce')

# Target components
targets = ['IG01', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04']

# Merge metric_window with global thresholds from metric_summary (global P95 and median)
mw = metric_window.merge(
    metric_summary[['cmdb_id','kpi_name','p95_value','median_value']],
    on=['cmdb_id','kpi_name'],
    how='left'
).copy()

# Filter only target components
mw = mw[mw['cmdb_id'].isin(targets)].copy()

# If no rows, create empty faults_table
if mw.empty:
    faults_table = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length',
        'max_value_in_run','severity_ratio','count_points_in_window','p95_value','median_value'
    ])
    faults_table_head = faults_table.head(20)
else:
    # Process each group to detect consecutive anomaly runs where value >= p95_value
    def tag_runs(g):
        g = g.sort_values('timestamp').copy()
        g['is_anom'] = g['value'] >= g['p95_value']
        g['prev_is_anom'] = g['is_anom'].shift(fill_value=False)
        # time gap: consider consecutive if timestamps differ by exactly 60 seconds
        g['time_gap'] = g['timestamp'].diff().fillna(60) != 60
        # start of a run: current is anomaly and (previous not anomaly OR time gap)
        g['start_run'] = g['is_anom'] & (~g['prev_is_anom'] | g['time_gap'])
        # cumulative run id (per group); multiply by is_anom to zero-out non-anomaly rows
        g['run_id'] = g['start_run'].cumsum() * g['is_anom']
        return g

    mw_tagged = mw.groupby(['cmdb_id','kpi_name'], group_keys=False).apply(tag_runs).reset_index(drop=True)

    # Aggregate runs where run_id > 0
    runs = mw_tagged[mw_tagged['run_id'] > 0].copy()
    if runs.empty:
        faults_table = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length',
            'max_value_in_run','severity_ratio','count_points_in_window','p95_value','median_value'
        ])
        faults_table_head = faults_table.head(20)
    else:
        # Per-run aggregation
        run_agg = runs.groupby(['cmdb_id','kpi_name','run_id'], as_index=False).agg(
            run_start_timestamp=('timestamp','min'),
            run_end_timestamp=('timestamp','max'),
            run_length=('timestamp','count'),
            max_value_in_run=('value','max'),
            p95_value=('p95_value','first'),
            median_value=('median_value','first')
        )

        # count_points_in_window per group (cmdb_id,kpi_name) from metric_window
        counts = mw.groupby(['cmdb_id','kpi_name'], as_index=False)['value'].count().rename(columns={'value':'count_points_in_window'})

        run_agg = run_agg.merge(counts, on=['cmdb_id','kpi_name'], how='left')

        # severity_ratio = max_value_in_run / p95_value (handle division by zero)
        run_agg['severity_ratio'] = run_agg.apply(
            lambda r: (r['max_value_in_run'] / r['p95_value']) if pd.notna(r['p95_value']) and r['p95_value'] != 0 else pd.NA,
            axis=1
        )

        # Keep only runs with run_length >= 2
        faults_table = run_agg[run_agg['run_length'] >= 2].copy()

        # Rounding for compactness
        faults_table['p95_value'] = faults_table['p95_value'].round(6)
        faults_table['median_value'] = faults_table['median_value'].round(6)
        faults_table['max_value_in_run'] = faults_table['max_value_in_run'].round(6)
        faults_table['severity_ratio'] = faults_table['severity_ratio'].round(6)

        # Order by run_length desc then severity_ratio desc
        faults_table = faults_table.sort_values(by=['run_length','severity_ratio'], ascending=[False, False]).reset_index(drop=True)

        # Save full runs table to kernel variable faults_table (already assigned)
        # Prepare compact head (top 20)
        faults_table_head = faults_table[[
            'cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp','run_length',
            'max_value_in_run','severity_ratio','count_points_in_window','p95_value','median_value'
        ]].head(20)

# Return compact table (top 20) and keep faults_table in kernel for later use
faults_table_head

```
Out[4]:
```
```python
# Build a concise plain-English summary of the detected consecutive anomaly runs.
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

try:
    faults_table
except NameError:
    summary = "No faults_table found in the kernel."
else:
    if faults_table.empty:
        summary = "No consecutive anomaly runs (length >= 2) detected for IG01 or Tomcat01-04 in the specified window."
    else:
        lines = []
        n = len(faults_table)
        lines.append(f"Detected {n} consecutive anomaly run(s) (run_length >= 2) for components IG01 and Tomcat01–04:")
        # limit to top 20 for compactness
        for _, r in faults_table.head(20).iterrows():
            start_ts = int(r['run_start_timestamp'])
            end_ts = int(r['run_end_timestamp'])
            start_local = datetime.fromtimestamp(start_ts, tz).strftime("%Y-%m-%d %H:%M:%S %Z")
            end_local = datetime.fromtimestamp(end_ts, tz).strftime("%Y-%m-%d %H:%M:%S %Z")
            lines.append(
                f"- {r['cmdb_id']} / {r['kpi_name']}: run_start={start_ts} ({start_local}), "
                f"run_end={end_ts} ({end_local}), run_length={int(r['run_length'])} points, "
                f"max_in_run={float(r['max_value_in_run']):,.6f}, p95={float(r['p95_value']):,.6f}, "
                f"median={float(r['median_value']):,.6f}, severity_ratio={float(r['severity_ratio']):.6f}, "
                f"points_in_window={int(r['count_points_in_window'])}"
            )
        summary = "\n".join(lines)

# Display the summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                     kpi_name  run_start_timestamp  run_end_timestamp  run_length  max_value_in_run  severity_ratio  count_points_in_window   p95_value  median_value
0    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed           1615266000         1615267800          31        57908672.0             1.0                      31  57908672.0    57908672.0```
```